In [1]:
from notebooks._utils import calculate_series_ensemble_accuracy
from notebooks._utils import calculate_parallel_ensemble_accuracy

ds_name = "myriadlama-debug"
dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"

def get_layers(model: str):
    if model.startswith("llama3.2_1b"):
        layers = [4, 8, 12, 16]
    elif model.startswith("llama3.2_3b"):
        layers = [7, 14, 21, 28]
    elif model.startswith("qwen2.5_3b"):
        layers = [9, 18, 27, 36]
    else:
        raise NotImplementedError(f"Layers not defined for model {model}")
    return layers


In [12]:
for model_name in ["llama3.2_1b"]:
    print(f"\n=================== Model: {model_name} ===================")
    dump_file_prefix = f"{dataset_root}/{ds_name}/{model_name}/myriadlama."    
    
    print("---- Calculating baseline ----")
    df = calculate_series_ensemble_accuracy(
        dump_file_prefix=dump_file_prefix, 
        single_para_qapair=True, explicit_prompts=False, repeat_paras=False, 
        modifyattn=False, modifyrope=False, scale_score=0, 
        num_paraphrases=1, num_fewshots=5)

    
    print("\n---- Logits-based Ensemble ----")
    calculate_parallel_ensemble_accuracy(
        dump_file_prefix=dump_file_prefix, 
        repeat_paras=False,
        logits_ensemble_method="avg",
        num_paraphrases=5, num_fewshots=5, use_generation=True)

    
    for layer in get_layers(model_name):
        for factor in [0.4, 0.6, 0.8]:
            print(f"\n---- Layer {layer} Ensemble (alpha={factor}) ----")
            calculate_parallel_ensemble_accuracy(
                dump_file_prefix=dump_file_prefix, 
                repeat_paras=False,
                logits_ensemble_method="avg",
                ensemble_method="ffn_activation", 
                ensemble_layer=layer, 
                multilayer=True, 
                ensemble_alpha=factor, 
                token_mode="all",
                num_paraphrases=5, num_fewshots=5, use_generation=True)
            
        


=================== Model: llama3.2_1b ===================
---- Calculating baseline ----
File ./singleparaqapair.5samples.1paras.feather does not exist!

---- Logits-based Ensemble ----
Acc: 0.4430 ==> 🏷️ 5paras 5shots  None layerNone  alpha1.0 token-all

---- Layer 4 Ensemble (alpha=0.4) ----
Acc: 0.4370 ==> 🏷️ 5paras 5shots  ffn_activation layer4 Multilayer alpha0.4 token-all

---- Layer 4 Ensemble (alpha=0.6) ----
Acc: 0.4100 ==> 🏷️ 5paras 5shots  ffn_activation layer4 Multilayer alpha0.6 token-all

---- Layer 4 Ensemble (alpha=0.8) ----
Acc: 0.3230 ==> 🏷️ 5paras 5shots  ffn_activation layer4 Multilayer alpha0.8 token-all

---- Layer 8 Ensemble (alpha=0.4) ----
Acc: 0.4370 ==> 🏷️ 5paras 5shots  ffn_activation layer8 Multilayer alpha0.4 token-all

---- Layer 8 Ensemble (alpha=0.6) ----
Acc: 0.4320 ==> 🏷️ 5paras 5shots  ffn_activation layer8 Multilayer alpha0.6 token-all

---- Layer 8 Ensemble (alpha=0.8) ----
Acc: 0.4180 ==> 🏷️ 5paras 5shots  ffn_activation layer8 Multilayer alpha0

In [4]:
dump_file_prefix

'/home/xzhao/workspace/GYB_self-ensemble/datasets/myriadlama/llama3.2_1b/myriadlama-debug.'